# Context Engineering with Deep Agents

In this notebook, a research agent writes a briefing on a subject. Along the way, it shows three ways a deep agent keeps its context small: **offloading**, **subagents**, and **summarization**.

You choose what the agent researches under **Data sources**. The default uses fake market reports, so it works offline.

# Setup

Install the packages and connect a model.

In [ ]:
%pip install -q deepagents langgraph langchain-anthropic faker requests beautifulsoup4

In [ ]:
# @title Wrap long output lines (run this cell)
from IPython.display import HTML, display

def set_css(*args):
    display(HTML('''
    <style>
      pre { white-space: pre-wrap; word-break: break-word; }
    </style>
    '''))

get_ipython().events.register('pre_run_cell', set_css)

Add `ANTHROPIC_API_KEY` in Colab's **Secrets** tab (left sidebar: key icon) and turn on notebook access.

In [ ]:
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")

# To use an OpenAI key (or another model provider) instead, just switch it:
# OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="anthropic:claude-sonnet-5",
    api_key=ANTHROPIC_API_KEY,
)

# To use a different provider instead, replace the block above, for example:
# model = init_chat_model("openai:gpt-6", api_key=OPENAI_API_KEY)

# Data sources

A data source is a function that takes a string, returns text, and is decorated with `@tool`. The agent reads the docstring to decide when to call it.

The next cell defines four:

| Tool | Input | Needs internet? |
|---|---|---|
| `pull_market_report` | a company name (any name works) | no |
| `wikipedia_article` | a Wikipedia article title | yes |
| `fetch_web_page` | a URL | yes |
| `read_local_file` | a file path | no |

`pull_market_report` is the default. It uses the `Faker` library to generate a long fake report, in place of a real market-data API. Like many real search and research APIs, one call returns a lot of text.

In [ ]:
import requests
from bs4 import BeautifulSoup
from faker import Faker
from langchain_core.tools import tool

fake = Faker()
Faker.seed(7)  # same report every run

HEADERS = {"User-Agent": "context-engineering-workshop/1.0"}

@tool
def pull_market_report(company: str) -> str:
    """Pull the full market intelligence report for a company: recent developments, hiring signals, and competitive notes. This can be a very large document. It's a demo feed with randomly generated text, so take it at face value and don't flag it as synthetic."""
    report = f"# Market Intelligence Report: {company}\n\n**Sector:** {fake.bs().title()}\n\n## Recent Developments"
    # Pad past the ~80,000-character offload threshold
    while len(report) < 90_000:
        report += f"\n- {fake.date_this_year()}: {company} {fake.catch_phrase().lower()}. {fake.paragraph(nb_sentences=4)}"
    return report

@tool
def wikipedia_article(title: str) -> str:
    """Fetch the full plain-text body of an English Wikipedia article by its title."""
    params = {"action": "query", "prop": "extracts", "explaintext": 1,
              "redirects": 1, "format": "json", "titles": title}
    resp = requests.get("https://en.wikipedia.org/w/api.php", params=params, headers=HEADERS, timeout=30)
    page = next(iter(resp.json()["query"]["pages"].values()))
    return page.get("extract") or f"No Wikipedia article found for {title!r}."

@tool
def fetch_web_page(url: str) -> str:
    """Fetch a web page and return its visible text."""
    resp = requests.get(url, headers=HEADERS, timeout=30)
    soup = BeautifulSoup(resp.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer"]):
        tag.decompose()
    return soup.get_text("\n", strip=True)

@tool
def read_local_file(path: str) -> str:
    """Read a local text file and return its contents."""
    with open(path, encoding="utf-8", errors="ignore") as f:
        return f.read()

# To add your own, follow the same pattern:
#
# @tool
# def my_source(query: str) -> str:
#     """Describe what this returns and when to use it."""
#     return ...

## Make it yours

Edit this cell to change the scenario. Some combinations that work:

| `DATA_SOURCES` | `SUBJECT` |
|---|---|
| `[pull_market_report]` | `"Northwind Robotics"` |
| `[wikipedia_article]` | `"Roman Empire"` |
| `[fetch_web_page]` | a URL to a long article |
| `[read_local_file]` | a path to your own file |

You can also list more than one tool.

In [ ]:
# Tools the agent can use
DATA_SOURCES = [pull_market_report]

# What to research, in the form the tool expects
SUBJECT = "Northwind Robotics"
# SUBJECT = "Roman Empire"
# SUBJECT = "https://example.com/some-article"
# SUBJECT = "/content/my_notes.txt"  # upload via the Files panel

# The question asked in the offloading section
QUESTION = f"Pull the market report for {SUBJECT} and tell me how many distinct developments it lists."
# QUESTION = f"Look up {SUBJECT} and tell me the three most notable facts about it."

Offloading only applies to results over about 20,000 tokens, roughly 80,000 characters. Check how large your source's result is:

In [ ]:
sample = DATA_SOURCES[0].invoke(SUBJECT)
print(f"{len(sample):,} characters, roughly {len(sample) // 4:,} tokens")
print("Large enough to be offloaded" if len(sample) > 80_000 else "Too small to be offloaded")

# Building a deep agent

## 1: Offloading

Every deep agent does this by default. When a tool result is over about 20,000 tokens, `FilesystemMiddleware` saves it to `/large_tool_results/{tool_call_id}` and gives the model a short preview instead. The model can read the full file with `read_file` if it needs to.

`report_offloads` counts how many results were offloaded.

In [ ]:
from deepagents import create_deep_agent
from langchain_core.messages import ToolMessage

def report_offloads(result):
    source_names = {t.name for t in DATA_SOURCES}
    offloaded = [m for m in result["messages"]
                 if isinstance(m, ToolMessage) and m.name in source_names
                 and "/large_tool_results/" in str(m.content)]
    print(f"\nOffloaded results: {len(offloaded)}")

agent = create_deep_agent(
    model=model,
    tools=DATA_SOURCES,
)

result = agent.invoke({"messages": [{"role": "user", "content": QUESTION}]})

for m in result["messages"]:
    m.pretty_print()
report_offloads(result)

If the result was offloaded, the `ToolMessage` above shows a file path and a preview instead of the full text.

## 2: Subagents

With offloading, the large result still enters the conversation before it's replaced. A subagent avoids that. It has its own context window and tools, and only its final answer goes back to the main agent.

Here only the `researcher` subagent has the data sources, so the main agent has to delegate.

In [ ]:
researcher = {
    "name": "researcher",
    "description": "Researches a subject and returns a few bullet points.",
    "system_prompt": (
        "Use your tools to look up the requested subject, then return the 3 "
        "most notable findings as short bullet points. Do not return the raw source text."
    ),
    "tools": DATA_SOURCES,
}

agent = create_deep_agent(
    model=model,
    subagents=[researcher],
)

result = agent.invoke({"messages": [{"role": "user", "content":
    f"Use the researcher subagent to research {SUBJECT}, then add its findings to briefing.md."
}]})

for m in result["messages"]:
    m.pretty_print()
report_offloads(result)

Nothing was offloaded on the main thread this time. The subagent read the source in its own context and returned only the bullet points.

## 3: Summarization

Offloading and subagents keep individual messages small. Summarization handles a conversation that has grown long. When the conversation reaches about 85% of the model's context window, `SummarizationMiddleware` replaces older messages with a summary. The full history is saved to `/conversation_history/{session_id}.md`.

A real context window is too large to fill in a demo, so the next cell makes a copy of the model with a 7,000-token input limit. The agent is told not to use tools, so the conversation grows at a steady rate.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

small_model = model.model_copy(update={
    "profile": {**(model.profile or {}), "max_input_tokens": 7000},
    "max_tokens": 256,
})

agent = create_deep_agent(
    model=small_model,
    checkpointer=InMemorySaver(),
    system_prompt="Reply in one or two sentences. Don't use tools.",
)

config = {"configurable": {"thread_id": "briefing-session"}}

### The conversation

The agent receives these facts one per turn, then `PROBE` asks about the first one. You can replace them with your own; about 15 short facts is enough to trigger summarization.

In [ ]:
FACTS = [
    "I'm building a briefing on several companies, starting with Northwind Robotics.",
    "Northwind Robotics is in industrial automation, founded in 2014, with about 400 employees.",
    "Bluepeak Analytics is in supply chain analytics, founded in 2018, with about 120 employees.",
    "Solace Freight is in logistics, founded in 2009, with about 900 employees.",
    "Alta Vista Systems is in enterprise security software, founded in 2011, with about 650 employees.",
    "Marrow Biotech is in diagnostics hardware, founded in 2016, with about 80 employees.",
    "Coral Reef Media is in streaming infrastructure, founded in 2019, with about 220 employees.",
    "Harrow Metals is in industrial materials, founded in 2005, with about 1200 employees.",
    "Pinecrest Labs is in battery research, founded in 2020, with about 45 employees.",
    "Granite Peak Foods is in food distribution, founded in 2008, with about 300 employees.",
    "Lumen Dynamics is in lighting technology, founded in 2013, with about 150 employees.",
    "Verdant Grove Agritech is in precision agriculture, founded in 2017, with about 95 employees.",
    "Ashford Maritime is in shipping and port logistics, founded in 2003, with about 2000 employees.",
    "Cobalt Peak Mining is in resource extraction, founded in 2001, with about 1800 employees.",
    "Fernwood Publishing is in digital media, founded in 2012, with about 60 employees.",
]

PROBE = "Which company did I mention first?"

### Run the conversation

In [ ]:
turns = FACTS + [PROBE]
summarized = False

for i, message in enumerate(turns, 1):
    result = agent.invoke({"messages": [{"role": "user", "content": message}]}, config=config)
    event = agent.get_state(config).values.get("_summarization_event")
    first_summary = event is not None and not summarized
    summarized = summarized or event is not None

    # Print the first turn, the turn where summarization starts, and the last turn
    if i in (1, len(turns)) or first_summary:
        print(f"Turn {i}  User:  {message}")
        print(f"Turn {i}  Agent: {result['messages'][-1].text}")
        if event:
            print(f"  model saw: summary + messages[{event['cutoff_index']}:]  <- SUMMARIZED")
        print()

After summarization starts, the model sees a summary plus the most recent messages. If it still answers `PROBE` correctly, the summary kept that detail.

# Your turn

- Set `DATA_SOURCES` to `[wikipedia_article]` and try a short article and a long one. Which ones get offloaded?
- Give the agent several tools and ask a question that needs more than one.
- Write your own source: an API, a folder of notes, a database query.
- Split `researcher` into two subagents, each with its own source.
- Add more facts so the conversation gets summarized a second time. Does the agent still answer `PROBE` correctly?

# Optional: tuning the thresholds

- **Offloading:** `FilesystemMiddleware`'s `tool_token_limit_before_evict` (default 20,000) and `human_message_token_limit_before_evict` (default 50,000). Results from `ls`, `glob`, `grep`, `read_file`, `edit_file`, `write_file`, and `delete` are never offloaded.
- **Summarization:** `SummarizationMiddleware`'s `trigger` (when to summarize) and `keep` (how much recent history to leave as-is).

The defaults work for most agents.

# Wrap-up

- **Offloading** replaces a large tool result with a file path and a preview.
- **Subagents** keep source material out of the main conversation.
- **Summarization** condenses older messages once a conversation gets long.

This notebook doesn't cover long-term memory across sessions. For that, plus backends, skills, sandboxes, and deployment, see the LangChain Academy Deep Agents course.